# Creating and modelling metallic supercells

In this section we will be concerned with modelling supercells of aluminium.
When dealing with periodic problems there is no unique definition of the
lattice: Clearly any duplication of the lattice along an axis is also a valid
repetitive unit to describe exactly the same system.
This is exactly what a **supercell** is: An $n$-fold repetition along one (or multiple)
axes of the original lattice.

The following code achieves this for aluminium:

In [1]:
using AtomsBuilder
using DFTK
using LinearAlgebra
using Unitful
using UnitfulAtomic
using PseudoPotentialData

function aluminium_setup(repeat=1; Ecut=7.0, kgrid=[2, 2, 2])
    # Use AtomsBuilder to setup aluminium cubic unit cell (4 Al atoms)
    # with provided lattice constant, see AtomsBase integration for details.
    unit_cell = bulk(:Al; a=7.65339u"bohr", cubic=true)
    supercell = unit_cell * (repeat, 1, 1)  # Make a supercell

    # Select standard pseudodojo pseudopotentials, construct an LDA model, discretize
    # Note: We disable symmetries explicitly here. Otherwise the problem sizes
    #       we are able to run on the CI are too simple to observe the numerical
    #       instabilities we want to trigger here.
    pseudopotentials = PseudoFamily("dojo.nc.sr.lda.v0_4_1.standard.upf")
    model = model_DFT(supercell; pseudopotentials, functionals=LDA(),
                      temperature=1e-3, symmetries=false)
    PlaneWaveBasis(model; Ecut, kgrid)
end;

As expected we obtain the unit cell for `repeat=1`:

In [2]:
aluminium_setup(1)

PlaneWaveBasis discretization:
    architecture         : DFTK.CPU()
    num. mpi processes   : 1
    num. julia threads   : 1
    num. DFTK  threads   : 1
    num. blas  threads   : 2
    num. fft   threads   : 1

    Ecut                 : 7.0 Ha
    fft_size             : (24, 24, 24), 13824 total points
    kgrid                : MonkhorstPack([2, 2, 2])
    num.   red. kpoints  : 8
    num. irred. kpoints  : 8

    Discretized Model(lda_x+lda_c_pw, 3D):
        lattice (in Bohr)    : [7.65339   , 0         , 0         ]
                               [0         , 7.65339   , 0         ]
                               [0         , 0         , 7.65339   ]
        unit cell volume     : 448.29 Bohr³
    
        atoms                : Al₄
        pseudopot. family    : PseudoFamily("dojo.nc.sr.lda.v0_4_1.standard.upf")
    
        num. electrons       : 12
        spin polarization    : none
        temperature          : 0.001 Ha
        smearing             : DFTK.Smearing.FermiDi

and 5-fold as large supercell with `repeat=5`:

In [3]:
aluminium_setup(5)

PlaneWaveBasis discretization:
    architecture         : DFTK.CPU()
    num. mpi processes   : 1
    num. julia threads   : 1
    num. DFTK  threads   : 1
    num. blas  threads   : 2
    num. fft   threads   : 1

    Ecut                 : 7.0 Ha
    fft_size             : (96, 24, 24), 55296 total points
    kgrid                : MonkhorstPack([2, 2, 2])
    num.   red. kpoints  : 8
    num. irred. kpoints  : 8

    Discretized Model(lda_x+lda_c_pw, 3D):
        lattice (in Bohr)    : [38.267    , 0         , 0         ]
                               [0         , 7.65339   , 0         ]
                               [0         , 0         , 7.65339   ]
        unit cell volume     : 2241.5 Bohr³
    
        atoms                : Al₂₀
        pseudopot. family    : PseudoFamily("dojo.nc.sr.lda.v0_4_1.standard.upf")
    
        num. electrons       : 60
        spin polarization    : none
        temperature          : 0.001 Ha
        smearing             : DFTK.Smearing.FermiD

As we will see in this notebook the modelling of a system generally becomes
harder if the system becomes larger.

- This sounds like a trivial statement as *per se* the cost per SCF step increases
  as the system (and thus $N$) gets larger.
- But there is more to it:
  If one is not careful also the *number of SCF iterations* increases
  as the system gets larger.
- The aim of a proper computational treatment of such supercells is therefore
  to ensure that the **number of SCF iterations remains constant** when the
  system size increases.

For achieving the latter DFTK by default employs the `LdosMixing`
preconditioner [^HL2021] during the SCF iterations. This mixing approach is
completely parameter free, but still automatically adapts to the treated
system in order to efficiently prevent charge sloshing. As a result,
modelling aluminium slabs indeed takes roughly the same number of SCF iterations
irrespective of the supercell size:

[^HL2021]:
   M. F. Herbst and A. Levitt.
   *Black-box inhomogeneous preconditioning for self-consistent field iterations in density functional theory.*
   J. Phys. Cond. Matt *33* 085503 (2021). [ArXiv:2009.01665](https://arxiv.org/abs/2009.01665)

In [4]:
self_consistent_field(aluminium_setup(1); tol=1e-4);

n     Energy            log10(ΔE)   log10(Δρ)   Diag   Δtime
---   ---------------   ---------   ---------   ----   ------
  1   -9.355201347315                   -1.09    6.0    156ms
  2   -9.356808739426       -2.79       -1.43    1.0   90.8ms
  3   -9.357066413377       -3.59       -2.77    2.4   96.5ms
  4   -9.357110900581       -4.35       -3.04    8.6    205ms
  5   -9.357111227333       -6.49       -3.20    1.2   83.7ms
  6   -9.357111368307       -6.85       -3.35    1.0   77.0ms
  7   -9.357111447805       -7.10       -3.50    2.2   91.8ms
  8   -9.357111485412       -7.42       -3.66    1.1   80.1ms
  9   -9.357111503292       -7.75       -3.92    1.5   88.7ms
 10   -9.357111507767       -8.35       -4.21    1.4   84.2ms


In [5]:
self_consistent_field(aluminium_setup(2); tol=1e-4);

n     Energy            log10(ΔE)   log10(Δρ)   Diag   Δtime
---   ---------------   ---------   ---------   ----   ------
  1   -18.74730108984                   -0.97    6.2    398ms
  2   -18.75459624450       -2.14       -1.34    1.8    231ms
  3   -18.79227702613       -1.42       -1.91    4.6    326ms
  4   -18.79256730969       -3.54       -2.16    3.8    307ms
  5   -18.79258186686       -4.84       -3.05    1.5    236ms
  6   -18.79260233920       -4.69       -3.33    8.2    475ms
  7   -18.79260612730       -5.42       -3.76    1.6    228ms
  8   -18.79260638360       -6.59       -4.15    2.6    285ms


In [6]:
self_consistent_field(aluminium_setup(4); tol=1e-4);

n     Energy            log10(ΔE)   log10(Δρ)   Diag   Δtime
---   ---------------   ---------   ---------   ----   ------
  1   -37.55104777543                   -0.84    9.4    1.69s
  2   -37.55761021952       -2.18       -1.22    1.8    801ms
  3   -37.55989426972       -2.64       -2.44    6.5    1.25s
  4   -37.56489840489       -2.30       -2.17   16.4    2.58s
  5   -37.56488362849   +   -4.83       -2.10    4.5    975ms
  6   -37.56494657148       -4.20       -2.78    1.0    797ms
  7   -37.56494924086       -5.57       -3.28    1.8    783ms
  8   -37.56494964333       -6.40       -3.66    7.0    1.24s
  9   -37.56494985451       -6.68       -4.17    2.8    1.02s


When switching off explicitly the `LdosMixing`, by selecting `mixing=SimpleMixing()`,
the performance of number of required SCF steps starts to increase as we increase
the size of the modelled problem:

In [7]:
self_consistent_field(aluminium_setup(1); tol=1e-4, mixing=SimpleMixing());

n     Energy            log10(ΔE)   log10(Δρ)   Diag   Δtime
---   ---------------   ---------   ---------   ----   ------
  1   -9.355139350376                   -1.10    6.0    194ms
  2   -9.356812631095       -2.78       -1.90    1.0   70.9ms
  3   -9.357002124589       -3.72       -2.24    4.5    136ms
  4   -9.357101594597       -4.00       -2.76    3.8    110ms
  5   -9.357111294507       -5.01       -3.70    2.0   93.6ms
  6   -9.357111411128       -6.93       -3.78    8.5    187ms
  7   -9.357111468254       -7.24       -3.95    1.1   74.5ms
  8   -9.357111507392       -7.41       -4.69    1.0   69.9ms


In [8]:
self_consistent_field(aluminium_setup(4); tol=1e-4, mixing=SimpleMixing());

n     Energy            log10(ΔE)   log10(Δρ)   Diag   Δtime
---   ---------------   ---------   ---------   ----   ------
  1   -37.54961085556                   -0.84    9.5    2.13s
  2   -37.54743405940   +   -2.66       -1.59    3.0    760ms
  3   -24.95976561085   +    1.10       -0.52    9.8    1.77s
  4   -37.51835586317        1.10       -1.61    8.9    1.72s
  5   -37.52573254462       -2.13       -1.72    1.8    799ms
  6   -37.37437821489   +   -0.82       -1.42    4.2    1.04s
  7   -37.54349459430       -0.77       -1.86    4.5    1.59s
  8   -37.56255017679       -1.72       -2.14    1.9    785ms
  9   -37.56392221461       -2.86       -2.24    2.9    1.02s
 10   -37.56475018710       -3.08       -2.71    1.9    822ms
 11   -37.56493117562       -3.74       -3.13    2.8    1.02s
 12   -37.56493897348       -5.11       -3.25    3.6    1.07s
 13   -37.56494615483       -5.14       -3.46    1.6    757ms
 14   -37.56494881953       -5.57       -3.69    2.1    813ms
 15   -37

For completion let us note that the more traditional `mixing=KerkerMixing()`
approach would also help in this particular setting to obtain a constant
number of SCF iterations for an increasing system size (try it!). In contrast
to `LdosMixing`, however, `KerkerMixing` is only suitable to model bulk metallic
system (like the case we are considering here). When modelling metallic surfaces
or mixtures of metals and insulators, `KerkerMixing` fails, while `LdosMixing`
still works well. See the Modelling a gallium arsenide surface example
or [^HL2021] for details. Due to the general applicability of `LdosMixing` this
method is the default mixing approach in DFTK.